<a href="https://colab.research.google.com/github/srihanreddy/Reinforcement-Learning/blob/main/LAB_PRACTICAL_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Assignment 1: Stochastic Policy in a Deterministic Environment

#### Formulating the Simultaneous Equations:
1. $V_\pi(S_0) = 0.9 \cdot Q_\pi(S_0, \text{Down}) + 0.1 \cdot Q_\pi(S_0, \text{Right})$
2. $V_\pi(S_1) = 0.9 \cdot Q_\pi(S_1, \text{Right}) + 0.1 \cdot Q_\pi(S_1, \text{Up})$

Where the action-values are:
- $Q_\pi(S_0, \text{Down}) = -1 + 0.9 \cdot V_\pi(S_1)$
- $Q_\pi(S_0, \text{Right}) = -1 + 0.9 \cdot V_\pi(S_0)$ (hitting a wall leaves agent in $S_0$ or is terminal state-like, but since the task says "reward for hitting wall = -1" and the transition is "S0 Right Wall", and the wall's future value is 0, we have $Q_\pi(S_0, \text{Right}) = -1 + 0.9 \cdot 0$)
- $Q_\pi(S_1, \text{Right}) = 100 + 0.9 \cdot V_\pi(G) = 100$
- $Q_\pi(S_1, \text{Up}) = -1 + 0.9 \cdot V_\pi(S_0)$

Substituting these gives:
- $V_\pi(S_0) = 0.9(-1 + 0.9V_\pi(S_1)) + 0.1(-1) = -1 + 0.81V_\pi(S_1)$
- $V_\pi(S_1) = 0.9(100) + 0.1(-1 + 0.9V_\pi(S_0)) = 89.9 + 0.09V_\pi(S_0)$

Solving these simultaneously:
- $V_\pi(S_0) \approx 77.21$
- $V_\pi(S_1) \approx 96.85$

In [1]:
import numpy as np

print("=== ASSIGNMENT 1: ALGEBRAIC SOLUTION ===")

# Coefficients from linear equations system:
#  1 * V(S0) - 0.81 * V(S1) = -1.0
# -0.09 * V(S0) + 1 * V(S1) = 89.9
A = np.array([
    [1.0, -0.81],
    [-0.09, 1.0]
])
B = np.array([-1.0, 89.9])

V_S0, V_S1 = np.linalg.solve(A, B)

# Calculate Action Values
Q_S0_down = -1.0 + 0.9 * V_S1
Q_S0_right = -1.0  # Hitting wall has transition to Wall (value=0)
Q_S1_right = 100.0  # Transition to G (value=0)
Q_S1_up = -1.0 + 0.9 * V_S0

# Verification of Bellman expectation
V_S0_check = 0.9 * Q_S0_down + 0.1 * Q_S0_right
V_S1_check = 0.9 * Q_S1_right + 0.1 * Q_S1_up

print(f"State Values:")
print(f"  V_pi(S0) = {V_S0:.2f}")
print(f"  V_pi(S1) = {V_S1:.2f}\n")

print(f"Action Values:")
print(f"  Q_pi(S0, Down)  = {Q_S0_down:.2f}")
print(f"  Q_pi(S0, Right) = {Q_S0_right:.2f}")
print(f"  Q_pi(S1, Right) = {Q_S1_right:.2f}")
print(f"  Q_pi(S1, Up)    = {Q_S1_up:.2f}\n")

print(f"Verification:")
print(f"  V_pi(S0) check: {V_S0_check:.2f} (Expected: {V_S0:.2f})")
print(f"  V_pi(S1) check: {V_S1_check:.2f} (Expected: {V_S1:.2f})")

=== ASSIGNMENT 1: ALGEBRAIC SOLUTION ===
State Values:
  V_pi(S0) = 77.47
  V_pi(S1) = 96.87

Action Values:
  Q_pi(S0, Down)  = 86.18
  Q_pi(S0, Right) = -1.00
  Q_pi(S1, Right) = 100.00
  Q_pi(S1, Up)    = 68.72

Verification:
  V_pi(S0) check: 77.47 (Expected: 77.47)
  V_pi(S1) check: 96.87 (Expected: 96.87)


### Assignment 2: Iterative Policy Evaluation (Stochastic Environment)

Under the policy $\pi(S_0) = \text{Down}$ and $\pi(S_1) = \text{Right}$, the state transitions are stochastic:
- **$S_0$ (Down)**: $0.1$ chance of Wall (reward $-1$, value $0$), $0.9$ chance of $S_1$ (reward $-1$)
- **$S_1$ (Right)**: $0.9$ chance of $G$ (reward $+100$, value $0$), $0.1$ chance of $S_0$ (reward $-1$)

In [2]:
print("=== ASSIGNMENT 2: ITERATIVE POLICY EVALUATION ===")

# Parameters
gamma = 0.9
theta = 0.000001

# Initializing state values
V_S0 = 0.0
V_S1 = 0.0

iterations = 0
while True:
    V_old_S0 = V_S0
    V_old_S1 = V_S1

    # Bellman update formula for stochastic transitions
    # V(S0) = 0.1 * [-1 + gamma*0] + 0.9 * [-1 + gamma*V(S1)]
    V_new_S0 = 0.1 * (-1) + 0.9 * (-1 + gamma * V_old_S1)

    # V(S1) = 0.9 * [100 + gamma*0] + 0.1 * [-1 + gamma*V(S0)]
    V_new_S1 = 0.9 * (100) + 0.1 * (-1 + gamma * V_old_S0)

    delta = max(abs(V_new_S0 - V_old_S0), abs(V_new_S1 - V_old_S1))

    V_S0 = V_new_S0
    V_S1 = V_new_S1
    iterations += 1

    if delta < theta:
        break

# Calculate all four Action Values after convergence
# Q(S0, Down) has the stochastic transition
Q_S0_down = 0.1 * (-1) + 0.9 * (-1 + gamma * V_S1)
# Q(S0, Right) is deterministic to Wall
Q_S0_right = 1.0 * (-1)
# Q(S1, Right) has the stochastic transition
Q_S1_right = 0.9 * (100) + 0.1 * (-1 + gamma * V_S0)
# Q(S1, Up) is deterministic to S0
Q_S1_up = 1.0 * (-1 + gamma * V_S0)

print(f"Number of evaluation iterations: {iterations}")
print(f"V_pi(S0) = {V_S0:.6f}")
print(f"V_pi(S1) = {V_S1:.6f}\n")

print("Action Values:")
print(f"  Q_pi(S0, Down)  = {Q_S0_down:.6f}")
print(f"  Q_pi(S0, Right) = {Q_S0_right:.6f}")
print(f"  Q_pi(S1, Right) = {Q_S1_right:.6f}")
print(f"  Q_pi(S1, Up)    = {Q_S1_up:.6f}\n")

print("Consistency Check:")
print(f"  V_pi(S0) == Q_pi(S0, Down)  : {abs(V_S0 - Q_S0_down) < 1e-9}")
print(f"  V_pi(S1) == Q_pi(S1, Right) : {abs(V_S1 - Q_S1_right) < 1e-9}")

=== ASSIGNMENT 2: ITERATIVE POLICY EVALUATION ===
Number of evaluation iterations: 15
V_pi(S0) = 77.466292
V_pi(S1) = 96.871966

Action Values:
  Q_pi(S0, Down)  = 77.466293
  Q_pi(S0, Right) = -1.000000
  Q_pi(S1, Right) = 96.871966
  Q_pi(S1, Up)    = 68.719663

Consistency Check:
  V_pi(S0) == Q_pi(S0, Down)  : False
  V_pi(S1) == Q_pi(S1, Right) : True
